In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '3_train_full_clean.parquet')

In [ ]:
# ============================================================
# 1. 강우사상별 데이터 분할 -> event_id 컬럼 생성
# ============================================================

# 1-1. 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    train_full[['timestamp']] 
    .drop_duplicates() 
    .sort_values('timestamp') 
    .reset_index(drop = True) 
)

# 1-2. diff 계산
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 1-3. 차분값이 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)

# 1-4. 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()

# 시험처가 제시한 강우사상 개수와 맞는지 확인: 122개
print(event_time['event_id'].nunique())

,timestamp,event_time_diff,event_id
0,2100-01-01 00:00:00,NaT,1
1,2100-01-01 01:00:00,0 days 01:00:00,1
2,2100-01-21 00:00:00,19 days 23:00:00,2
3,2100-01-21 01:00:00,0 days 01:00:00,2
4,2100-01-21 02:00:00,0 days 01:00:00,2


122


In [17]:
# ============================================================
# 2. event_id 기준 데이터 분할
# ============================================================

# 2-1. train_full에 event_id 컬럼 붙이기(매핑)
rule1 = event_time.set_index('timestamp')['event_id']  # 매핑규칙(조인대상 인덱스 필수)

train_full['event_id'] = train_full['timestamp'].map(rule1)

# event_id 98번 기준으로 분할(98번은 train에 배치)
split_num = 98
train = train_full[train_full['event_id'] <= split_num]
val = train_full[train_full['event_id'] > split_num]

print("분할 전후 행열개수")
print(f"분할 전: {train_full.shape}")
print(f"분할 후: (train){train.shape}, (val){val.shape}")

분할 전후 행열개수
분할 전: (749609, 35)
분할 후: (train)(616293, 35), (val)(133316, 35)


In [18]:
# ============================================================
# 4. 데이터셋 저장하기
# ============================================================

train.to_parquet(PROJECT_ROOT / 'data' / '4_train_split.parquet')
val.to_parquet(PROJECT_ROOT / 'data' / '4_val_split.parquet')